In [1]:
import pathlib, sys
import matplotlib
matplotlib.use('Agg')
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# 06 · Decision Framework

The arena's verdict, turned into a rule you can apply — and the two utilities (`clean` and `evaluate_cleaning`) that let you measure rather than assume on your own data.

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from src.cleaning.pipelines import PIPELINE_NAMES, clean
from src.datasets.corpora import load_sentiment
from src.evaluation.comparison import evaluate_cleaning
from src.models.bow import BowClassifier
from src.visualisation import save_figure, save_json
pd.set_option('display.max_colwidth', 70)

## The `clean(text, pipeline)` utility

One call, four named pipelines, so you can see the spectrum on your own text.

In [3]:
demo = 'The US release of Apple Vision Pro was NOT a flop, I would never doubt them!'
clean_demo = pd.DataFrame([{'pipeline': p, 'result': clean(demo, p)} for p in PIPELINE_NAMES])
save_json({'text': demo, 'pipelines': clean_demo.to_dict(orient='records')}, 'nb06_clean_demo.json')
clean_demo

,pipeline,result
0,none,"The US release of Apple Vision Pro was NOT a flop, I would never d..."
1,minimal,"The US release of Apple Vision Pro was NOT a flop, I would never d..."
2,standard,"release apple vision pro flop, would doubt"
3,aggressive,releas appl vision pro flop would doubt


## The `evaluate_cleaning(texts, labels, model)` harness

Give it your data and a way to build your model; it runs every pipeline, cross-validated, and tells you which cleaning actually wins — the meta-rule in code form.

In [4]:
texts, labels = load_sentiment()
scoreboard = evaluate_cleaning(texts, labels, BowClassifier)
save_json(scoreboard, 'nb06_evaluate_cleaning_demo.json')
scoreboard.round(3)

,model,pipeline,accuracy,macro_f1,n_splits
0,tfidf+logreg,none,0.988,0.987,5
1,tfidf+logreg,minimal,0.988,0.987,5
2,tfidf+logreg,standard,0.975,0.975,5
3,tfidf+logreg,aggressive,0.962,0.962,5


## The rules

In [5]:
rules = [
    {'if': 'Classic bag-of-words / TF-IDF model', 'then': 'Standard cleaning usually helps — apply it.'},
    {'if': 'Transformer or any pretrained model', 'then': 'Minimal cleaning only; keep casing, stopwords, punctuation.'},
    {'if': 'Sparse retrieval (BM25)', 'then': 'Stemming and normalization help (more term overlap).'},
    {'if': 'Dense retrieval', 'then': 'Feed raw text the encoder expects.'},
    {'if': 'Sentiment / negation / tone carries the label', 'then': 'Never remove stopwords (it deletes not/no/never).'},
    {'if': 'Any step, any model', 'then': 'Measure it against raw text first — cleaning is irreversible.'},
]
save_json(rules, 'nb06_decision_rules.json')
pd.DataFrame(rules)

,if,then
0,Classic bag-of-words / TF-IDF model,Standard cleaning usually helps — apply it.
1,Transformer or any pretrained model,"Minimal cleaning only; keep casing, stopwords, punctuation."
2,Sparse retrieval (BM25),Stemming and normalization help (more term overlap).
3,Dense retrieval,Feed raw text the encoder expects.
4,Sentiment / negation / tone carries the label,Never remove stopwords (it deletes not/no/never).
5,"Any step, any model",Measure it against raw text first — cleaning is irreversible.


In [6]:
def box(ax, x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle='round,pad=0.02',
                                linewidth=1.2, edgecolor='#333333', facecolor=color))
    ax.text(x, y, text, ha='center', va='center', fontsize=9, wrap=True)

def arrow(ax, x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=12,
                                 color='#555555', linewidth=1.0))

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
box(ax, 5, 9.2, 5.6, 0.9, 'Before cleaning:\nwhat model and task?', '#dce6f2')
box(ax, 2.5, 7, 3.6, 0.9, 'Classic / TF-IDF / BoW', '#e8f0e3')
box(ax, 7.5, 7, 3.6, 0.9, 'Transformer / pretrained', '#f2e6dc')
box(ax, 2.5, 5, 3.9, 1.0, 'Standard cleaning helps\n(lowercase, stopwords, lemmatize)', '#e8f0e3')
box(ax, 7.5, 5, 3.9, 1.0, 'Minimal cleaning only\n(keep casing, stopwords, punctuation)', '#f2e6dc')
box(ax, 2.5, 2.9, 3.9, 1.0, 'Retrieval with BM25:\nstemming & normalization help', '#e8f0e3')
box(ax, 7.5, 2.9, 3.9, 1.0, 'Dense retrieval:\nfeed raw text', '#f2e6dc')
box(ax, 5, 0.9, 9.4, 1.0, 'Meta-rule: never apply a step without measuring it against raw text '
    'on YOUR model and task. Cleaning is irreversible.', '#f7e9c3')
arrow(ax, 4.2, 8.8, 2.9, 7.5); arrow(ax, 5.8, 8.8, 7.1, 7.5)
arrow(ax, 2.5, 6.55, 2.5, 5.55); arrow(ax, 7.5, 6.55, 7.5, 5.55)
arrow(ax, 2.5, 4.5, 2.5, 3.45); arrow(ax, 7.5, 4.5, 7.5, 3.45)
fig.tight_layout()
save_figure(fig, 'nb06_decision_framework.png')

## The close

Minimal cleaning for modern models, standard cleaning only for bag-of-words, stemming for sparse retrieval and raw text for dense. But the rule that outranks all of them is the harness above: the cleaning ritual that opens every NLP notebook is optimized for models you probably are not using, and the only way to know what helps *your* model is to measure it against raw text — before you throw the information away for good.